# Lifecycle security: the scorecard's calibration, and the gap between policy and practice

**Claim cluster:** `C062`, `C063`, `C071`, `C072`, `C073`, `C074`, `C102`, `C104`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C062', 'C063', 'C071', 'C072', 'C073', 'C074', 'C102', 'C104')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Wave 1 charted stated support periods by device category. This notebook looks at the other half of the same subject: what the scorecard is calibrated to measure, and which practical gaps the corpus records around it. The claims below run from documented facts (`C062`, `C063`, `C074`) to reported signals about specific findings (`C071`, `C072`, `C073`, `C102`, `C104`), so the panels keep the two apart.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### The scorecard's own calibration, before any product is scored

`M066` records 14 lifecycle controls, `M067` and `M068` record the highest and lowest control weights, and `M064`/`M065` record the composite scale: a maximum of 3.00 and a passing score of 2.00 on a weighted average. That is the framework the corpus proposes. The database holds no per-control scores, so this figure shows *calibration*, not results, and the uncertainty section says so explicitly.

In [ ]:
# Framework rows, read by id. M064/M065 share the unit "weighted average", so
# they may share an axis; M066-M068 are counts and weights and are annotated
# instead of plotted on that axis.
_scale = viz.metrics_by_id(["M064", "M065"])
_scale["reading"] = ["composite maximum", "minimum passing score"]
show(_scale[["local_id", "reading", "value", "unit", "confidence",
             "source_ref"]].rename(
    columns={"local_id": "metric", "source_ref": "source",
             "confidence": "conf"}))

_structure = viz.metrics_by_id(["M066", "M067", "M068"])
show(_structure[["local_id", "metric_name", "value", "unit", "confidence",
                 "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))

_passing = viz.numeric_value(_scale.loc[_scale["local_id"] == "M065", "value"].iloc[0])
fig = go.Figure(go.Bar(
    x=[viz.numeric_value(value) for value in _scale["value"]],
    y=_scale["reading"] + " (" + _scale["local_id"] + ")",
    orientation="h",
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in _scale["confidence"]],
    text=["%g" % viz.numeric_value(value) for value in _scale["value"]],
    textposition="outside",
    customdata=list(zip(_scale["value"], _scale["unit"], _scale["source_ref"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]} %{customdata[1]}"
                   "<br>source: %{customdata[2]}<extra></extra>"),
))
if _passing is not None:
    fig.add_vline(x=_passing, line_dash="dot", line_color="#333333",
                  annotation_text="M065: minimum passing score",
                  annotation_position="top right")
fig.update_layout(
    title=("The scorecard's calibration as recorded: 14 controls, weights "
           "1.0 and 0.7, composite out of 3.00 (M064-M068)"),
    xaxis_title="weighted-average composite",
    height=320,
    yaxis=dict(autorange="reversed"),
)
fig.show()

## Supporting context

### Where the corpus's lifecycle-security evidence actually sits

The count below is computed, not estimated: every claim recorded against workflow stage `W9` (Lifecycle Security and Privacy Scoring), grouped by the section it was written in. It answers a different question from the scorecard: not 'how secure is a product' but 'which lifecycle-security subjects did this corpus spend its evidence on'. Sections are document sections, not control areas.

In [ ]:
_corpus_claims = viz.fetch_all_claims()
_w9 = (_corpus_claims[_corpus_claims["workflow_stage"] == "W9"]
       .groupby("section").size().reset_index(name="claims")
       .sort_values("claims", ascending=False))
display(HTML(
    f"<h4>W9 claims by section ({int(_w9['claims'].sum())} claims in total)</h4>"
    "<p>Computed by grouping the full claim table; no count is typed in by "
    "hand.</p>"
))
show(_w9)
fig = go.Figure(go.Bar(
    x=_w9["claims"],
    y=_w9["section"],
    orientation="h",
    marker_color="#4c78a8",
    text=_w9["claims"],
    textposition="outside",
    hovertemplate="%{y}<br>%{x} claims at stage W9<extra></extra>",
))
fig.update_layout(
    title=("Where the corpus put its lifecycle-security claims "
           "(workflow stage W9, by section)"),
    xaxis_title="claims",
    height=380,
    yaxis=dict(autorange="reversed", automargin=True),
)
fig.show()

# The practical gaps that are recorded as numbers, each with its own kind of
# unit: a count of product models, a duration in months, and an explicit "none".
display(HTML(
    "<h4>Recorded practice gaps</h4>"
    "<p><code>M069</code> counts product models found transmitting video in "
    "plaintext; <code>M063</code> records AI-assistant data-retention "
    "defaults; <code>M070</code> records that UK PSTI mandates no minimum "
    "support period at all, which is a different statement from mandating a "
    "short one.</p>"
))
show(viz.metrics_by_id(["M069", "M063", "M070"])[
    ["local_id", "metric_name", "value", "unit", "confidence", "source_ref",
     "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### This page cannot rank any product

The corpus stores the scorecard's shape (`M064`-`M068`) and no per-control scores, so a 14-control radar or a product league table would be invented rather than drawn. Wave 1 reached the same conclusion from the support-period side; this notebook reaches it from the calibration side.

Other limits on this page:

- **The section counts measure documentation, not security.** The chart above counts where claims were written, which is a statement about this corpus's attention, not about any product's lifecycle controls.
- **`M070` cannot be drawn as a zero bar.** UK PSTI mandating no minimum period is a different statement from mandating a short one, so it stays a table row.
- **`M069` is two product models from one publication.** It is a finding, not an incidence rate, and the corpus records no sample frame behind it.
- **`C071` (prompt injection with physical consequences) and `C072` (biometric spoofing with no liveness standard) are reported signals at medium confidence.** Both describe demonstrated attacks without a measured base rate.
- **`C074` is the counterweight and is kept visible.** Matter secure commissioning (PASE) and device attestation are recorded as robust and mandatory for certified devices, so this page is not a claim that nothing works.
- **`C102` records adoption without enforcement.** Vulnerability disclosure programmes are spreading, but the corpus records no response SLAs, so the claim is described rather than charted.

In [ ]:
# The framework rows and the practice findings are separated here, because one
# is a proposed measurement instrument and the others are observations.
display(HTML(
    "<h4>Framework or finding? Each claim in the cluster, labelled</h4>"
    "<p>A claim that defines a control is not evidence that the control is "
    "met.</p>"
))
_framework_claims = ["C062", "C063", "C104"]
show(claims.loc[claims["local_id"].isin(_framework_claims),
                ["local_id", "claim_type", "confidence", "workflow_stage",
                 "source_refs", "falsifier_missing", "claim_text"]].rename(
    columns={"local_id": "claim", "claim_type": "type", "confidence": "conf",
             "workflow_stage": "stage", "source_refs": "sources",
             "falsifier_missing": "no_falsifier"}))
_finding_claims = ["C071", "C072", "C073", "C074", "C102"]
show(claims.loc[claims["local_id"].isin(_finding_claims),
                ["local_id", "claim_type", "confidence", "workflow_stage",
                 "source_refs", "falsifier_missing", "claim_text"]].rename(
    columns={"local_id": "claim", "claim_type": "type", "confidence": "conf",
             "workflow_stage": "stage", "source_refs": "sources",
             "falsifier_missing": "no_falsifier"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Framework rows and practice findings are read by explicit metric id with `viz.metrics_by_id`; the section counts come from `viz.fetch_all_claims` grouped in pandas. This notebook deliberately reuses no figure from `notebooks/1/15_support_period_landscape.ipynb`, which covers mandated and stated support periods.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()